In [ ]:
# 🚀 DÍA 27: POLICY GRADIENT + SELF-SUPERVISED LEARNING

# 1. Instalación de dependencias
!pip install -q gymnasium numpy matplotlib seaborn tensorflow torch sentence-transformers

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow: {tf.__version__}")
print("="*70)
print("🎓 DÍA 27: POLICY GRADIENT + SELF-SUPERVISED LEARNING")
print("="*70)


# 2. Teoría Rápida
print("""
CONCEPTOS CLAVE:

🎯 POLICY GRADIENT (RL Avanzado):
- En lugar de aprender Q-values, el agente aprende una POLÍTICA π(a|s)
- La política es una función que dice: "en estado s, haz acción a con probabilidad p"
- Es el método detrás de RLHF (lo que usó ChatGPT para alinearse)
- Algoritmo REINFORCE: el más simple de Policy Gradient

🧠 SELF-SUPERVISED LEARNING:
- La IA crea sus propias etiquetas a partir de los datos
- Ejemplo: "predice la palabra faltante en esta oración"
- Así se entrenó GPT-4 con billones de textos sin etiquetar
- SimCLR: versión para imágenes (contrastive learning)

💡 DIFERENCIA CLAVE:
- Supervised: "Este es un gato" (etiqueta humana)
- Self-Supervised: "Completa esta oración" (la IA se auto-evalúa)
""")


# Parte 1: Policy Gradient (REINFORCE)
print("\n" + "="*70)
print("🎮 PARTE 1: POLICY GRADIENT CON CARTPOLE")
print("="*70)

print("""
🎯 REINFORCE Algorithm:
1. El agente juega un episodio completo
2. Al final, calcula el retorno total (recompensa acumulada)
3. Si la acción fue buena (alta recompensa), la refuerza
4. Si fue mala, la debilita
5. Repite miles de veces
""")

# Crear entorno
env = gym.make('CartPole-v1')

print(f"\n Entorno CartPole:")
print(f"   Estados: {env.observation_space.shape}")
print(f"   Acciones: {env.action_space.n}")


# 3. Crear Red Política
print("\n" + "="*70)
print("Creando Red de Política (Policy Network)")
print("="*70)

state_size = int(env.observation_space.shape[0])
action_size = int(env.action_space.n)

def build_policy_network():
    """Red que output probabilidades de cada acción"""
    model = Sequential([
        Input(shape=(state_size,)),
        Dense(24, activation='relu'),
        Dense(24, activation='relu'),
        Dense(action_size, activation='softmax')  # ✅ Softmax = probabilidades
    ])
    model.compile(optimizer=Adam(learning_rate=0.01), loss='categorical_crossentropy')
    return model

policy_net = build_policy_network()
print("\n Arquitectura de la Policy Network:")
policy_net.summary()


# 4. Entrenar con Reinforce
print("\n" + "="*70)
print("Entrenando con Reinforce (Policy Gradient)")
print("="*70)

EPISODES = 100
GAMMA = 0.99
rewards_history = []

print(f"\nEntrenando por {EPISODES} episodios...\n")

for episode in range(EPISODES):
    state, _ = env.reset()
    episode_states = []
    episode_actions = []
    episode_rewards = []
    total_reward = 0
    done = False

    # Jugar episodio completo
    while not done:
        # Predecir probabilidades de acciones
        state_input = state.reshape(1, -1)
        action_probs = policy_net.predict(state_input, verbose=0)[0]

        # Elegir acción según la política (muestreo)
        action = np.random.choice(action_size, p=action_probs)

        # Ejecutar acción
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Guardar experiencia
        episode_states.append(state)
        episode_actions.append(action)
        episode_rewards.append(reward)

        state = next_state
        total_reward += reward

    rewards_history.append(total_reward)

    # Calcular retornos descontados (discounted returns)
    discounted_rewards = []
    cumulative = 0
    for r in reversed(episode_rewards):
        cumulative = r + GAMMA * cumulative
        discounted_rewards.insert(0, cumulative)

    # Normalizar retornos
    discounted_rewards = np.array(discounted_rewards)
    if len(discounted_rewards) > 1:
        discounted_rewards = (discounted_rewards - np.mean(discounted_rewards)) / (np.std(discounted_rewards) + 1e-9)

    # Entrenar la política
    states = np.array(episode_states)
    actions = np.zeros((len(episode_actions), action_size))
    for i, a in enumerate(episode_actions):
        actions[i][a] = 1  # One-hot encoding

    # Pesos = retornos descontados (refuerza buenas acciones)
    sample_weights = discounted_rewards

    policy_net.fit(states, actions, sample_weight=sample_weights,
                   epochs=1, verbose=0)

    # Progreso
    if (episode + 1) % 10 == 0:
        avg = np.mean(rewards_history[-10:])
        print(f"   Episodio {episode+1}/{EPISODES} | Recompensa promedio (últimos 10): {avg:.1f}")

print(f"\n✅ Entrenamiento Policy Gradient completado")


# 5. Visualizar Aprendizaje
print("\n" + "="*70)
print("Visualización de Aprendizaje")
print("="*70)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Recompensa por episodio
axes[0].plot(rewards_history, alpha=0.3, color='blue', label='Recompensa')
if len(rewards_history) >= 10:
    axes[0].plot(np.convolve(rewards_history, np.ones(10)/10, mode='valid'),
                 color='red', linewidth=2, label='Promedio móvil (10)')
axes[0].axhline(y=200, color='green', linestyle='--', label='Meta: 200')
axes[0].set_title('Policy Gradient: Recompensa por Episodio', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Episodio')
axes[0].set_ylabel('Recompensa')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribución
axes[1].hist(rewards_history, bins=20, color='purple', edgecolor='black', alpha=0.7)
axes[1].axvline(x=np.mean(rewards_history), color='red', linestyle='--',
                label=f'Media: {np.mean(rewards_history):.1f}')
axes[1].set_title('Distribución de Recompensas', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Recompensa')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# 6. Probar Agente Entrenando
print("\n" + "="*70)
print("Probando Agente con Política Aprendida")
print("="*70)

print("\n Ejecutando 5 episodios con la política entrenada:\n")

for episode in range(5):
    state, _ = env.reset()
    total_reward = 0
    done = False
    steps = 0

    while not done:
        state_input = state.reshape(1, -1)
        action_probs = policy_net.predict(state_input, verbose=0)[0]
        action = np.argmax(action_probs)  # ✅ Siempre la mejor acción

        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward
        steps += 1

    status = "✅ ÉXITO" if total_reward >= 200 else "⚠️ Parcial"
    print(f"   Episodio {episode+1}: {total_reward} pasos - {status}")

print(f"\n Promedio: {np.mean(rewards_history[-10:]):.1f} pasos")


# Parte 2: Self-Supervised Learning
print("\n" + "="*70)
print("Parte 2: Self-Supervised Learning")
print("="*70)

print("""
¿Qué es Self-Supervised Learning?

La IA aprende SIN etiquetas humanas creando sus propios "ejercicios":

📝 Para TEXTO (como GPT):
   "El gato está en el [MASK]" → La IA predice "techo"
   No necesita que nadie le diga qué es un gato

🖼️ Para IMÁGENES (SimCLR):
   Toma una imagen, la transforma 2 veces (zoom, rotación)
   La IA aprende que ambas versiones son "la misma imagen"
   Así aprende representaciones visuales sin etiquetas

🎯 RESULTADO:
   Modelos que entienden el mundo sin supervisión humana
   GPT-4, DALL-E, CLIP = Self-Supervised Learning
""")


# 7. Demo: Embeddings Auto-Supervisados
print("\n" + "="*70)
print("Demo: Embeddings de Texto con Self-Supervised Learning")
print("="*70)

print("\n Cargando modelo pre-entrenado (sentence-transformers)...")
print("   (Primera vez tarda 1-2 minutos en descargar ~400MB)\n")

from sentence_transformers import SentenceTransformer

# Modelo entrenado con self-supervised learning
model_ssl = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Modelo cargado")

# Frases de ejemplo
frases = [
    "El perro juega en el parque",
    "Un canino corre en el jardín",
    "La computadora procesa datos",
    "El ordenador analiza información",
    "Me gusta comer pizza los viernes",
    "Los gatos duermen mucho durante el día"
]

print(f"\n Frases a analizar:")
for i, frase in enumerate(frases, 1):
    print(f"   {i}. {frase}")

# Generar embeddings (representaciones vectoriales)
print("\n Generando embeddings (vectores de 384 dimensiones)...")
embeddings = model_ssl.encode(frases)

print(f"   Shape de embeddings: {embeddings.shape}")
print(f"   Cada frase = vector de {embeddings.shape[1]} números")


# 8. Calcular Similitudes
print("\n" + "="*70)
print("Matriz de Similitud Semántica")
print("="*70)

from sklearn.metrics.pairwise import cosine_similarity

# Calcular similitud coseno entre todas las frases
similarity_matrix = cosine_similarity(embeddings)

print("\n Similitudes (1.0 = idénticas, 0.0 = sin relación):\n")

# Mostrar matriz
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(similarity_matrix,
            xticklabels=[f"{i+1}" for i in range(len(frases))],
            yticklabels=[f"{i+1}" for i in range(len(frases))],
            annot=True, fmt=".2f", cmap='YlOrRd',
            square=True, ax=ax)
ax.set_title('Matriz de Similitud Semántica\n(Self-Supervised Learning)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Frase')
ax.set_ylabel('Frase')
plt.tight_layout()
plt.show()

# Mostrar pares más similares
print("\n Pares de frases más similares:\n")

pares_similares = []
for i in range(len(frases)):
    for j in range(i+1, len(frases)):
        sim = similarity_matrix[i][j]
        pares_similares.append((sim, frases[i], frases[j]))

pares_similares.sort(reverse=True)

for sim, f1, f2 in pares_similares[:5]:
    print(f"   Similitud: {sim:.3f}")
    print(f"   📝 '{f1}'")
    print(f"   📝 '{f2}'")
    print()


# 9. Visualizar en 2D(PCA)
print("\n" + "="*70)
print("Visualización de Embeddings en 2D")
print("="*70)

from sklearn.decomposition import PCA

# Reducir a 2 dimensiones
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(12, 8))

colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4', '#ffeaa7', '#dda0dd']
for i, (x, y) in enumerate(embeddings_2d):
    ax.scatter(x, y, color=colors[i], s=200, edgecolors='black', linewidth=2, zorder=5)
    ax.annotate(frases[i][:30] + "...",
                (x, y), fontsize=9, fontweight='bold',
                xytext=(10, 10), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=colors[i], alpha=0.7))

ax.set_title(' Embeddings en 2D (PCA)\nFrases semánticamente similares están cerca',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Componente 1')
ax.set_ylabel('Componente 2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# 10. Lecciones Clave
print("\n" + "="*70)
print("Lecciones Clave del día 27")
print("="*70)

print("""
1. Policy Gradient (REINFORCE):
   - El agente aprende una POLÍTICA π(a|s) directamente
   - No necesita Q-table ni valores de estado
   - Es más estable que DQN en algunos casos
   - Base de algoritmos modernos: PPO, A3C, SAC

2. Diferencia Q-Learning vs Policy Gradient:
   - Q-Learning: "¿Cuánto vale esta acción?" → indirecto
   - Policy Gradient: "¿Qué acción debo tomar?" → directo
   - ChatGPT usa Policy Gradient (RLHF) para alinearse

3. Self-Supervised Learning:
   - La IA se auto-entrena sin etiquetas humanas
   - GPT: predice la siguiente palabra
   - SimCLR: compara versiones transformadas de imágenes
   - Es la base de los LLMs modernos (GPT-4, Claude, Gemini)

4. Embeddings:
   - Representaciones vectoriales de datos
   - Frases similares → vectores cercanos
   - Permiten búsqueda semántica, recomendación, etc.

5. Por qué importa:
   - Los datos etiquetados son caros y escasos
   - Los datos sin etiquetar son abundantes (todo internet)
   - Self-Supervised = el futuro del ML
""")

print("\n✅ ¡DÍA 27 COMPLETADO! ")
